SECTION 1: SETUP AND LOAD PROCESSED DATA

In [ ]:
import pandas as pd
import numpy as np
import os
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import KNNImputer

import matplotlib.pyplot as plt
import seaborn as sns


SECTION 2: CONFIGURATION

In [ ]:
DATA_DIR = r"C:\Users\snehi\OneDrive - California State University Chico\MATH699P\Data"
PROCESSED_DIR = os.path.join(DATA_DIR, 'processed_data')

print(f"Input Directory: {PROCESSED_DIR}")
print("\nLoading processed data...")
print("-" * 80)

try:
    df_ozone = pd.read_csv(
        os.path.join(PROCESSED_DIR, 'ozone_processed.csv'),
        parse_dates=['DATE_TIME']
    )
    print(f"Loaded ozone data: {df_ozone.shape}")
except Exception as e:
    print(f"Error loading ozone data: {str(e)}")
    print("Make sure you ran the data extraction notebook first!")
    raise



In [ ]:

# Load other datasets
try:
    df_gas = pd.read_csv(
        os.path.join(PROCESSED_DIR, 'hourly_gas_processed.csv'),
        parse_dates=['DATE_TIME']
    )
    print(f"Loaded gas data: {df_gas.shape}")
except:
    df_gas = pd.DataFrame()
    print("No gas data available")

try:
    df_site = pd.read_csv(os.path.join(PROCESSED_DIR, 'site_metadata.csv'))
    print(f"Loaded site metadata: {df_site.shape}")
except:
    df_site = pd.DataFrame()
    print("No site metadata available")


cols_to_drop = ['OZONE_F', 'UNITS', 'EXPECTED_VALUE', 'CALIBRATION_VALUE', 
                'CALIBRATION_TYPE', 'UPDATE_DATE']
df_ozone = df_ozone.drop(columns=[c for c in cols_to_drop if c in df_ozone.columns])

SECTION 3: TEMPORAL FEATURE ENGINEERING

In [ ]:
class TemporalFeatureEngineer:
    """Create time-based features"""
    
    def __init__(self):
        pass
    
    def add_temporal_features(self, df, datetime_col='DATE_TIME'):
        """Add comprehensive temporal features"""
        
        print("\nAdding temporal features...")
        print("-" * 80)
        
        df = df.copy()
        
        # Ensure datetime
        if datetime_col not in df.columns:
            print(f"Column {datetime_col} not found")
            return df
            
        df[datetime_col] = pd.to_datetime(df[datetime_col])
        
        # Basic temporal features
        df['year'] = df[datetime_col].dt.year
        df['month'] = df[datetime_col].dt.month
        df['day'] = df[datetime_col].dt.day
        df['hour'] = df[datetime_col].dt.hour
        df['dayofweek'] = df[datetime_col].dt.dayofweek  # Monday=0, Sunday=6
        df['dayofyear'] = df[datetime_col].dt.dayofyear
        df['week'] = df[datetime_col].dt.isocalendar().week.astype(int)
        df['quarter'] = df[datetime_col].dt.quarter
        
        # Weekend indicator
        df['is_weekend'] = df['dayofweek'].isin([5, 6]).astype(int)
        
        # Time of day categories
        df['hour_category'] = pd.cut(
            df['hour'],
            bins=[-1, 6, 12, 18, 24],
            labels=['night', 'morning', 'afternoon', 'evening']
        )
        
        # Rush hour indicator
        df['is_rush_hour'] = df['hour'].isin([7, 8, 9, 17, 18, 19]).astype(int)
        
        # Season
        df['season'] = df['month'].map({
            12: 'winter', 1: 'winter', 2: 'winter',
            3: 'spring', 4: 'spring', 5: 'spring',
            6: 'summer', 7: 'summer', 8: 'summer',
            9: 'fall', 10: 'fall', 11: 'fall'
        })
        
        # Cyclical encoding for periodic features
        # Hour (24-hour cycle)
        df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
        df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
        
        # Month (12-month cycle)
        df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
        df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
        
        # Day of week (7-day cycle)
        df['dayofweek_sin'] = np.sin(2 * np.pi * df['dayofweek'] / 7)
        df['dayofweek_cos'] = np.cos(2 * np.pi * df['dayofweek'] / 7)
        
        # Day of year (365-day cycle)
        df['dayofyear_sin'] = np.sin(2 * np.pi * df['dayofyear'] / 365.25)
        df['dayofyear_cos'] = np.cos(2 * np.pi * df['dayofyear'] / 365.25)
        
        new_features = [c for c in df.columns if c not in [datetime_col, 'SITE_ID', 'OZONE']]
        print(f"Added {len(new_features)} temporal features")
        
        return df


SECTION 4: LAG AND ROLLING FEATURES

In [ ]:
class LagRollingFeatureEngineer:
    """Create lag and rolling window features"""
    
    def __init__(self):
        pass
    
    def create_lag_features(self, df, target_col, lags=[1, 2, 3, 6, 12, 24], 
                          group_col='SITE_ID'):
        """Create lag features"""
        
        if target_col not in df.columns:
            print(f"Target column {target_col} not found")
            return df
        
        print(f"\nCreating lag features for {target_col}...")
        print(f"   Lags: {lags}")
        print("-" * 80)
        
        df = df.copy()
        
        if group_col in df.columns:
            for lag in lags:
                col_name = f'{target_col}_lag_{lag}'
                df[col_name] = df.groupby(group_col)[target_col].shift(lag)
        else:
            for lag in lags:
                col_name = f'{target_col}_lag_{lag}'
                df[col_name] = df[target_col].shift(lag)
            
        print(f"Created {len(lags)} lag features")
        
        return df
    
    def create_rolling_features(self, df, target_col, windows=[3, 6, 12, 24, 168],
                               group_col='SITE_ID'):
        """Create rolling window statistics"""
        
        if target_col not in df.columns:
            print(f"Target column {target_col} not found")
            return df
        
        print(f"\nCreating rolling features for {target_col}...")
        print(f"   Windows: {windows}")
        print("-" * 80)
        
        df = df.copy()
        
        if group_col in df.columns:
            for window in windows:
                # Rolling mean
                df[f'{target_col}_rolling_mean_{window}'] = df.groupby(group_col)[target_col].transform(
                    lambda x: x.rolling(window=window, min_periods=1).mean()
                )
                
                # Rolling std
                df[f'{target_col}_rolling_std_{window}'] = df.groupby(group_col)[target_col].transform(
                    lambda x: x.rolling(window=window, min_periods=1).std()
                )
                
                # Rolling min
                df[f'{target_col}_rolling_min_{window}'] = df.groupby(group_col)[target_col].transform(
                    lambda x: x.rolling(window=window, min_periods=1).min()
                )
                
                # Rolling max
                df[f'{target_col}_rolling_max_{window}'] = df.groupby(group_col)[target_col].transform(
                    lambda x: x.rolling(window=window, min_periods=1).max()
                )
                
                # Rolling range
                df[f'{target_col}_rolling_range_{window}'] = (
                    df[f'{target_col}_rolling_max_{window}'] - 
                    df[f'{target_col}_rolling_min_{window}']
                )
        else:
            for window in windows:
                df[f'{target_col}_rolling_mean_{window}'] = df[target_col].rolling(window=window, min_periods=1).mean()
                df[f'{target_col}_rolling_std_{window}'] = df[target_col].rolling(window=window, min_periods=1).std()
        
        print(f"Created {len(windows) * 5} rolling features")
        
        return df
    
    def create_diff_features(self, df, target_col, periods=[1, 24, 168],
                           group_col='SITE_ID'):
        """Create differencing features"""
        
        if target_col not in df.columns:
            print(f"Target column {target_col} not found")
            return df
        
        print(f"\nCreating differencing features for {target_col}...")
        print(f"   Periods: {periods}")
        print("-" * 80)
        
        df = df.copy()
        
        if group_col in df.columns:
            for period in periods:
                df[f'{target_col}_diff_{period}'] = df.groupby(group_col)[target_col].diff(periods=period)
        else:
            for period in periods:
                df[f'{target_col}_diff_{period}'] = df[target_col].diff(periods=period)
        
        print(f"Created {len(periods)} differencing features")
        
        return df
    
    def create_rate_of_change(self, df, target_col, group_col='SITE_ID'):
        """Calculate rate of change (velocity and acceleration)"""
        
        if target_col not in df.columns:
            print(f"Target column {target_col} not found")
            return df
        
        print(f"\nCreating rate of change features for {target_col}...")
        print("-" * 80)
        
        df = df.copy()
        
        # First derivative (velocity)
        if group_col in df.columns:
            df[f'{target_col}_velocity'] = df.groupby(group_col)[target_col].diff(1)
            df[f'{target_col}_acceleration'] = df.groupby(group_col)[f'{target_col}_velocity'].diff(1)
        else:
            df[f'{target_col}_velocity'] = df[target_col].diff(1)
            df[f'{target_col}_acceleration'] = df[f'{target_col}_velocity'].diff(1)
        
        print(f"Created 2 rate of change features")
        
        return df


SECTION 5: STATISTICAL FEATURES

In [ ]:
class StatisticalFeatureEngineer:
    """Create statistical features"""
    
    def __init__(self):
        pass
    
    def create_expanding_features(self, df, target_col, group_col='SITE_ID'):
        """Create expanding window features (cumulative statistics)"""
        
        if target_col not in df.columns:
            print(f"Target column {target_col} not found")
            return df
        
        print(f"\nCreating expanding window features for {target_col}...")
        print("-" * 80)
        
        df = df.copy()
        
        if group_col in df.columns:
            # Expanding mean
            df[f'{target_col}_expanding_mean'] = df.groupby(group_col)[target_col].transform(
                lambda x: x.expanding(min_periods=1).mean()
            )
            
            # Expanding std
            df[f'{target_col}_expanding_std'] = df.groupby(group_col)[target_col].transform(
                lambda x: x.expanding(min_periods=1).std()
            )
        else:
            df[f'{target_col}_expanding_mean'] = df[target_col].expanding(min_periods=1).mean()
            df[f'{target_col}_expanding_std'] = df[target_col].expanding(min_periods=1).std()
        
        print(f"Created 2 expanding window features")
        
        return df
    
    def create_percentile_features(self, df, target_col, group_col='SITE_ID'):
        """Create percentile-based features"""
        
        if target_col not in df.columns or group_col not in df.columns:
            print(f"Required columns not found")
            return df
        
        print(f"\nCreating percentile features for {target_col}...")
        print("-" * 80)
        
        df = df.copy()
        
        # Site-level percentiles
        site_stats = df.groupby(group_col)[target_col].agg([
            ('p25', lambda x: x.quantile(0.25)),
            ('p50', lambda x: x.quantile(0.50)),
            ('p75', lambda x: x.quantile(0.75)),
            ('p90', lambda x: x.quantile(0.90))
        ]).reset_index()
        
        df = df.merge(site_stats, on=group_col, how='left', suffixes=('', '_site'))
        
        # Relative position within site distribution
        if target_col in df.columns:
            df[f'{target_col}_above_site_median'] = (df[target_col] > df['p50']).astype(int)
            df[f'{target_col}_above_site_p75'] = (df[target_col] > df['p75']).astype(int)
        
        print(f"Created percentile features")
        
        return df


SECTION 6: MISSING DATA HANDLING

In [ ]:
class MissingDataHandler:
    """Handle missing data"""
    
    def __init__(self):
        pass
    
    def analyze_missing(self, df):
        """Analyze missing data patterns"""
        
        print("\nMISSING DATA ANALYSIS")
        print("="*80)
        
        # Get numeric columns
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        
        missing_info = []
        
        for col in numeric_cols:
            missing_count = df[col].isna().sum()
            missing_pct = (missing_count / len(df)) * 100
            
            if missing_count > 0:
                missing_info.append({
                    'Column': col,
                    'Missing Count': missing_count,
                    'Missing %': missing_pct
                })
        
        if missing_info:
            missing_df = pd.DataFrame(missing_info).sort_values('Missing %', ascending=False)
            print(missing_df.head(20).to_string(index=False))
            if len(missing_df) > 20:
                print(f"\n... and {len(missing_df) - 20} more columns with missing values")
        else:
            print("No missing values found!")
        
        return missing_info
    
    def create_complete_time_index(self, df, site_col='SITE_ID', 
                                   time_col='DATE_TIME', freq='H'):
        """Create complete time index for each site"""
        
        if site_col not in df.columns or time_col not in df.columns:
            print(f"Required columns not found")
            return df
        
        print(f"\nCreating complete time index (freq={freq})...")
        print("-" * 80)
        
        result_dfs = []
        
        sites = df[site_col].unique()
        print(f"Processing {len(sites)} sites...")
        
        for i, site in enumerate(sites):
            if (i + 1) % 10 == 0 or i == 0:
                print(f"   Processing site {i+1}/{len(sites)}")
            
            site_data = df[df[site_col] == site].copy()
            
            # Create complete time range
            start = site_data[time_col].min()
            end = site_data[time_col].max()
            complete_index = pd.date_range(start=start, end=end, freq=freq)
            
            # Reindex
            site_data = site_data.set_index(time_col)
            site_data = site_data.reindex(complete_index)
            site_data[site_col] = site
            site_data.index.name = time_col
            site_data = site_data.reset_index()
            
            result_dfs.append(site_data)
        
        df_complete = pd.concat(result_dfs, ignore_index=True)
        
        print(f"Complete index created: {len(df_complete):,} records")
        
        return df_complete
    
    def interpolate_missing(self, df, method='linear', limit=3):
        """Interpolate missing values IN-PLACE to save memory"""
        
        print(f"\nInterpolating missing values (method={method}, limit={limit})...")
        print("-" * 80)
        
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        
        before_missing = df[numeric_cols].isna().sum().sum()
        
        # Process column by column IN-PLACE
        for col in numeric_cols:
            if df[col].isna().any():
                df[col] = df[col].interpolate(
                    method='linear',
                    limit=limit,
                    limit_direction='both'
                ).astype('float32')  # Keep as float32
        
        after_missing = df[numeric_cols].isna().sum().sum()
        
        print(f"Before: {before_missing:,} missing values")
        print(f"After:  {after_missing:,} missing values")
        print(f"Filled: {before_missing - after_missing:,} values")
        
        return df


SECTION 7: OUTLIER DETECTION AND HANDLING

In [ ]:
class OutlierHandler:
    """Detect and handle outliers"""
    
    def __init__(self):
        pass
    
    def detect_outliers_iqr(self, df, columns, multiplier=3.0):
        """Detect outliers using IQR method"""
        
        print(f"\nDetecting outliers (IQR method, multiplier={multiplier})...")
        print("-" * 80)
        
        outlier_mask = pd.Series(False, index=df.index)
        outlier_info = {}
        
        for col in columns:
            if col not in df.columns:
                continue
            
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            
            lower_bound = Q1 - multiplier * IQR
            upper_bound = Q3 + multiplier * IQR
            
            col_outliers = (df[col] < lower_bound) | (df[col] > upper_bound)
            outlier_mask = outlier_mask | col_outliers
            
            outlier_count = col_outliers.sum()
            outlier_pct = (outlier_count / len(df)) * 100
            
            outlier_info[col] = {
                'count': outlier_count,
                'percentage': outlier_pct,
                'lower_bound': lower_bound,
                'upper_bound': upper_bound
            }
            
            print(f"{col:30s}: {outlier_count:6d} outliers ({outlier_pct:5.2f}%)")
        
        total_outliers = outlier_mask.sum()
        print(f"\n   Total rows with outliers: {total_outliers:,} ({total_outliers/len(df)*100:.2f}%)")
        
        return outlier_mask, outlier_info
    
    def handle_outliers(self, df, outlier_mask, method='clip'):
        """Handle outliers"""
        
        print(f"\nHandling outliers (method={method})...")
        print("-" * 80)
        
        df_clean = df.copy()
        
        if method == 'remove':
            df_clean = df_clean[~outlier_mask]
            print(f"   Removed {outlier_mask.sum():,} rows")
        
        elif method == 'nan':
            numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
            df_clean.loc[outlier_mask, numeric_cols] = np.nan
            print(f"   Set {outlier_mask.sum():,} rows to NaN")
        
        elif method == 'clip':
            numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
            for col in numeric_cols:
                if df_clean[col].notna().any():
                    p01 = df_clean[col].quantile(0.01)
                    p99 = df_clean[col].quantile(0.99)
                    df_clean[col] = df_clean[col].clip(lower=p01, upper=p99)
            print(f"Clipped values to 1st-99th percentile range")
        
        return df_clean


SECTION 8: EXECUTE FEATURE ENGINEERING PIPELINE

In [ ]:
import gc

# Initialize feature engineers
temporal_fe = TemporalFeatureEngineer()
lag_rolling_fe = LagRollingFeatureEngineer()
statistical_fe = StatisticalFeatureEngineer()
missing_handler = MissingDataHandler()
outlier_handler = OutlierHandler()

# Drop useless columns first
cols_to_drop = ['OZONE_F', 'UNITS', 'EXPECTED_VALUE', 'CALIBRATION_VALUE', 
                'CALIBRATION_TYPE', 'UPDATE_DATE']
df_ozone = df_ozone.drop(columns=[c for c in cols_to_drop if c in df_ozone.columns])

# Downcast to float32
float_cols = df_ozone.select_dtypes(include=['float64']).columns
df_ozone[float_cols] = df_ozone[float_cols].astype('float32')

print(f"Starting shape: {df_ozone.shape}")

# Process site by site and save to parquet files
OUTPUT_CHUNKS_DIR = os.path.join(PROCESSED_DIR, 'feature_chunks')
os.makedirs(OUTPUT_CHUNKS_DIR, exist_ok=True)

sites = df_ozone['SITE_ID'].unique()
print(f"Processing {len(sites)} sites...")

for i, site_id in enumerate(sites):
    print(f"\n[{i+1}/{len(sites)}] Processing site: {site_id}")
    
    # Extract site data
    site_df = df_ozone[df_ozone['SITE_ID'] == site_id].copy()
    
    # Step 1: Temporal features
    site_df = temporal_fe.add_temporal_features(site_df)
    
    # Step 2: Create complete time index for this site
    start = site_df['DATE_TIME'].min()
    end = site_df['DATE_TIME'].max()
    complete_index = pd.date_range(start=start, end=end, freq='h')
    
    site_df = site_df.set_index('DATE_TIME')
    site_df = site_df.reindex(complete_index)
    site_df['SITE_ID'] = site_id
    site_df.index.name = 'DATE_TIME'
    site_df = site_df.reset_index()
    
    # Step 3: Lag features (no groupby needed - single site)
    if 'OZONE' in site_df.columns:
        for lag in [1, 2, 3, 6, 12, 24, 48, 168]:
            site_df[f'OZONE_lag_{lag}'] = site_df['OZONE'].shift(lag).astype('float32')
    
    # Step 4: Rolling features
    if 'OZONE' in site_df.columns:
        for window in [3, 6, 12, 24, 168]:
            site_df[f'OZONE_rolling_mean_{window}'] = site_df['OZONE'].rolling(window, min_periods=1).mean().astype('float32')
            site_df[f'OZONE_rolling_std_{window}'] = site_df['OZONE'].rolling(window, min_periods=1).std().astype('float32')
            site_df[f'OZONE_rolling_min_{window}'] = site_df['OZONE'].rolling(window, min_periods=1).min().astype('float32')
            site_df[f'OZONE_rolling_max_{window}'] = site_df['OZONE'].rolling(window, min_periods=1).max().astype('float32')
            site_df[f'OZONE_rolling_range_{window}'] = (
                site_df[f'OZONE_rolling_max_{window}'] - site_df[f'OZONE_rolling_min_{window}']
            ).astype('float32')
    
    # Step 5: Diff features
    if 'OZONE' in site_df.columns:
        for period in [1, 24, 168]:
            site_df[f'OZONE_diff_{period}'] = site_df['OZONE'].diff(period).astype('float32')
        site_df['OZONE_velocity'] = site_df['OZONE'].diff(1).astype('float32')
        site_df['OZONE_acceleration'] = site_df['OZONE_velocity'].diff(1).astype('float32')
    
    # Step 6: Interpolate missing (in-place for this site)
    numeric_cols = site_df.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        if site_df[col].isna().any():
            site_df[col] = site_df[col].interpolate(method='linear', limit=3, limit_direction='both')
    
    # Step 7: Clip outliers for OZONE
    if 'OZONE' in site_df.columns:
        p01 = site_df['OZONE'].quantile(0.01)
        p99 = site_df['OZONE'].quantile(0.99)
        site_df['OZONE'] = site_df['OZONE'].clip(lower=p01, upper=p99)
    
    # Save this site's features to parquet (much more efficient than CSV)
    output_file = os.path.join(OUTPUT_CHUNKS_DIR, f'features_{site_id}.parquet')
    site_df.to_parquet(output_file, index=False)
    
    # Clear memory
    del site_df
    gc.collect()

print("\n" + "="*80)
print("All sites processed! Combining chunks...")

# Combine all parquet files
all_files = [os.path.join(OUTPUT_CHUNKS_DIR, f) for f in os.listdir(OUTPUT_CHUNKS_DIR) if f.endswith('.parquet')]
df_features = pd.concat([pd.read_parquet(f) for f in all_files], ignore_index=True)

# Add site metadata
if not df_site.empty:
    site_features = ['SITE_ID', 'LATITUDE', 'LONGITUDE', 'ELEVATION', 'LAND_USE', 'TERRAIN', 'STATE']
    available_site_features = [f for f in site_features if f in df_site.columns]
    if available_site_features:
        df_features = df_features.merge(df_site[available_site_features], on='SITE_ID', how='left')

print(f"Final shape: {df_features.shape}")

SECTION 9: SAVE ENGINEERED FEATURES

In [ ]:
# Save full feature set
output_file = os.path.join(PROCESSED_DIR, 'features_engineered.csv')
df_features.to_csv(output_file, index=False)
print(f"✅ Saved: {output_file}")

# Save feature list
feature_list = df_features.columns.tolist()
feature_list_file = os.path.join(PROCESSED_DIR, 'feature_list.txt')
with open(feature_list_file, 'w') as f:
    f.write("FEATURE LIST\n")
    f.write("="*80 + "\n\n")
    f.write(f"Total Features: {len(feature_list)}\n")
    f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
    
    # Categorize features
    temporal_features = [f for f in feature_list if any(x in f.lower() for x in ['year', 'month', 'day', 'hour', 'week', 'season', 'sin', 'cos', 'weekend', 'rush'])]
    lag_features = [f for f in feature_list if 'lag' in f.lower()]
    rolling_features = [f for f in feature_list if 'rolling' in f.lower()]
    diff_features = [f for f in feature_list if 'diff' in f.lower() or 'velocity' in f.lower() or 'acceleration' in f.lower()]
    stat_features = [f for f in feature_list if any(x in f.lower() for x in ['expanding', 'percentile', 'p25', 'p50', 'p75', 'p90', 'above'])]
    site_features = [f for f in feature_list if any(x in f.lower() for x in ['latitude', 'longitude', 'elevation', 'land', 'terrain', 'state'])]
    
    f.write(f"\nTEMPORAL FEATURES ({len(temporal_features)}):\n")
    for feat in temporal_features:
        f.write(f"  • {feat}\n")
    
    f.write(f"\nLAG FEATURES ({len(lag_features)}):\n")
    for feat in lag_features:
        f.write(f"  • {feat}\n")
    
    f.write(f"\nROLLING WINDOW FEATURES ({len(rolling_features)}):\n")
    for feat in rolling_features:
        f.write(f"  • {feat}\n")
    
    f.write(f"\nDIFFERENCING FEATURES ({len(diff_features)}):\n")
    for feat in diff_features:
        f.write(f"  • {feat}\n")
    
    f.write(f"\nSTATISTICAL FEATURES ({len(stat_features)}):\n")
    for feat in stat_features:
        f.write(f"  • {feat}\n")
    
    f.write(f"\nSITE FEATURES ({len(site_features)}):\n")
    for feat in site_features:
        f.write(f"  • {feat}\n")
    
    f.write(f"\nOTHER FEATURES:\n")
    other_features = [f for f in feature_list if f not in temporal_features + lag_features + rolling_features + diff_features + stat_features + site_features]
    for feat in other_features:
        f.write(f"  • {feat}\n")

print(f"✅ Saved feature list: {feature_list_file}")

print(f"\n✅ All files saved to: {PROCESSED_DIR}")
print(f"\nNext step: Run the train/test split notebook")
print(f"\nScript completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

In [1]:
import pandas as pd
import numpy as np
import os
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

import dask.dataframe as dd
from dask.diagnostics import ProgressBar
import gc

# =============================================================================
# CONFIGURATION
# =============================================================================
DATA_DIR = r"C:\Users\snehi\OneDrive - California State University Chico\MATH699P\Data"
PROCESSED_DIR = os.path.join(DATA_DIR, 'processed_data')
OUTPUT_DIR = os.path.join(PROCESSED_DIR, 'features_parquet')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Input Directory: {PROCESSED_DIR}")
print(f"Output Directory: {OUTPUT_DIR}")

# =============================================================================
# LOAD DATA
# =============================================================================
print("\nLoading processed data...")
print("-" * 80)

df_ozone = pd.read_csv(
    os.path.join(PROCESSED_DIR, 'ozone_processed.csv'),
    parse_dates=['DATE_TIME']
)
print(f"Loaded ozone data: {df_ozone.shape}")

# Load site metadata
try:
    df_site = pd.read_csv(os.path.join(PROCESSED_DIR, 'site_metadata.csv'))
    print(f"Loaded site metadata: {df_site.shape}")
except:
    df_site = pd.DataFrame()
    print("No site metadata available")

# =============================================================================
# PREPROCESSING - Drop useless columns and downcast
# =============================================================================
print("\nPreprocessing...")
print("-" * 80)

# Drop columns with 100% missing or not needed
cols_to_drop = ['OZONE_F', 'UNITS', 'EXPECTED_VALUE', 'CALIBRATION_VALUE', 
                'CALIBRATION_TYPE', 'UPDATE_DATE']
df_ozone = df_ozone.drop(columns=[c for c in cols_to_drop if c in df_ozone.columns])
print(f"Dropped unnecessary columns. New shape: {df_ozone.shape}")

# Downcast to float32
float_cols = df_ozone.select_dtypes(include=['float64']).columns
df_ozone[float_cols] = df_ozone[float_cols].astype('float32')

int_cols = df_ozone.select_dtypes(include=['int64']).columns
df_ozone[int_cols] = df_ozone[int_cols].astype('int32')

print(f"Memory usage after downcast: {df_ozone.memory_usage(deep=True).sum() / 1e9:.2f} GB")

# =============================================================================
# FEATURE ENGINEERING FUNCTIONS
# =============================================================================

def add_temporal_features(df):
    """Add time-based features to a pandas DataFrame"""
    df = df.copy()
    
    # Basic temporal features
    df['year'] = df['DATE_TIME'].dt.year.astype('int16')
    df['month'] = df['DATE_TIME'].dt.month.astype('int8')
    df['day'] = df['DATE_TIME'].dt.day.astype('int8')
    df['hour'] = df['DATE_TIME'].dt.hour.astype('int8')
    df['dayofweek'] = df['DATE_TIME'].dt.dayofweek.astype('int8')
    df['dayofyear'] = df['DATE_TIME'].dt.dayofyear.astype('int16')
    df['week'] = df['DATE_TIME'].dt.isocalendar().week.astype('int8')
    df['quarter'] = df['DATE_TIME'].dt.quarter.astype('int8')
    
    # Weekend indicator
    df['is_weekend'] = (df['dayofweek'] >= 5).astype('int8')
    
    # Rush hour indicator (7-9 AM and 5-7 PM)
    df['is_rush_hour'] = df['hour'].isin([7, 8, 9, 17, 18, 19]).astype('int8')
    
    # Season
    season_map = {
        12: 0, 1: 0, 2: 0,   # winter = 0
        3: 1, 4: 1, 5: 1,    # spring = 1
        6: 2, 7: 2, 8: 2,    # summer = 2
        9: 3, 10: 3, 11: 3   # fall = 3
    }
    df['season'] = df['month'].map(season_map).astype('int8')
    
    # Cyclical encoding
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24).astype('float32')
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24).astype('float32')
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12).astype('float32')
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12).astype('float32')
    df['dayofweek_sin'] = np.sin(2 * np.pi * df['dayofweek'] / 7).astype('float32')
    df['dayofweek_cos'] = np.cos(2 * np.pi * df['dayofweek'] / 7).astype('float32')
    df['dayofyear_sin'] = np.sin(2 * np.pi * df['dayofyear'] / 365.25).astype('float32')
    df['dayofyear_cos'] = np.cos(2 * np.pi * df['dayofyear'] / 365.25).astype('float32')
    
    return df


def process_site_features(site_df):
    """
    Process all features for a single site.
    This function is applied to each site's data separately.
    """
    site_df = site_df.sort_values('DATE_TIME').copy()
    
    # Skip if no OZONE data
    if 'OZONE' not in site_df.columns or site_df['OZONE'].isna().all():
        return site_df
    
    # --- LAG FEATURES ---
    lags = [1, 2, 3, 6, 12, 24, 48, 168]
    for lag in lags:
        site_df[f'OZONE_lag_{lag}'] = site_df['OZONE'].shift(lag).astype('float32')
    
    # --- ROLLING FEATURES ---
    windows = [3, 6, 12, 24, 168]
    for window in windows:
        rolling = site_df['OZONE'].rolling(window=window, min_periods=1)
        site_df[f'OZONE_rolling_mean_{window}'] = rolling.mean().astype('float32')
        site_df[f'OZONE_rolling_std_{window}'] = rolling.std().astype('float32')
        site_df[f'OZONE_rolling_min_{window}'] = rolling.min().astype('float32')
        site_df[f'OZONE_rolling_max_{window}'] = rolling.max().astype('float32')
        site_df[f'OZONE_rolling_range_{window}'] = (
            site_df[f'OZONE_rolling_max_{window}'] - site_df[f'OZONE_rolling_min_{window}']
        ).astype('float32')
    
    # --- DIFFERENCING FEATURES ---
    periods = [1, 24, 168]
    for period in periods:
        site_df[f'OZONE_diff_{period}'] = site_df['OZONE'].diff(period).astype('float32')
    
    # --- RATE OF CHANGE ---
    site_df['OZONE_velocity'] = site_df['OZONE'].diff(1).astype('float32')
    site_df['OZONE_acceleration'] = site_df['OZONE_velocity'].diff(1).astype('float32')
    
    # --- EXPANDING FEATURES ---
    site_df['OZONE_expanding_mean'] = site_df['OZONE'].expanding(min_periods=1).mean().astype('float32')
    site_df['OZONE_expanding_std'] = site_df['OZONE'].expanding(min_periods=1).std().astype('float32')
    
    # --- PERCENTILE FEATURES (site-level) ---
    p25 = site_df['OZONE'].quantile(0.25)
    p50 = site_df['OZONE'].quantile(0.50)
    p75 = site_df['OZONE'].quantile(0.75)
    p90 = site_df['OZONE'].quantile(0.90)
    
    site_df['OZONE_site_p25'] = p25
    site_df['OZONE_site_p50'] = p50
    site_df['OZONE_site_p75'] = p75
    site_df['OZONE_site_p90'] = p90
    site_df['OZONE_above_site_median'] = (site_df['OZONE'] > p50).astype('int8')
    site_df['OZONE_above_site_p75'] = (site_df['OZONE'] > p75).astype('int8')
    
    # --- INTERPOLATE MISSING VALUES ---
    numeric_cols = site_df.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        if site_df[col].isna().any():
            site_df[col] = site_df[col].interpolate(
                method='linear', 
                limit=3, 
                limit_direction='both'
            )
    
    # --- CLIP OUTLIERS ---
    if site_df['OZONE'].notna().any():
        p01 = site_df['OZONE'].quantile(0.01)
        p99 = site_df['OZONE'].quantile(0.99)
        site_df['OZONE'] = site_df['OZONE'].clip(lower=p01, upper=p99)
    
    return site_df


# =============================================================================
# MAIN PROCESSING PIPELINE WITH DASK
# =============================================================================
print("\n" + "=" * 80)
print("STARTING FEATURE ENGINEERING WITH DASK")
print("=" * 80)

# Step 1: Add temporal features (can be done on full dataframe)
print("\nStep 1: Adding temporal features...")
df_ozone = add_temporal_features(df_ozone)
print(f"   Added temporal features. Shape: {df_ozone.shape}")

# Step 2: Sort by site and time
print("\nStep 2: Sorting data...")
df_ozone = df_ozone.sort_values(['SITE_ID', 'DATE_TIME']).reset_index(drop=True)

# Step 3: Convert to Dask DataFrame
print("\nStep 3: Converting to Dask DataFrame...")
# Partition by approximately 500k rows each
n_partitions = max(1, len(df_ozone) // 500_000)
ddf = dd.from_pandas(df_ozone, npartitions=n_partitions)
print(f"   Created Dask DataFrame with {n_partitions} partitions")

# Clear pandas DataFrame to free memory
del df_ozone
gc.collect()

# Step 4: Apply site-level feature engineering using groupby + apply
print("\nStep 4: Engineering site-level features (lag, rolling, diff, etc.)...")
print("   This may take a while...")

# Define the meta (output schema) for the apply function
# We need to specify all the columns that will be in the output
meta_cols = {
    'SITE_ID': 'object',
    'DATE_TIME': 'datetime64[ns]',
    'OZONE': 'float32',
    'QA_CODE': 'int32',
    'year': 'int16',
    'month': 'int8',
    'day': 'int8',
    'hour': 'int8',
    'dayofweek': 'int8',
    'dayofyear': 'int16',
    'week': 'int8',
    'quarter': 'int8',
    'is_weekend': 'int8',
    'is_rush_hour': 'int8',
    'season': 'int8',
    'hour_sin': 'float32',
    'hour_cos': 'float32',
    'month_sin': 'float32',
    'month_cos': 'float32',
    'dayofweek_sin': 'float32',
    'dayofweek_cos': 'float32',
    'dayofyear_sin': 'float32',
    'dayofyear_cos': 'float32',
}

# Add lag features to meta
for lag in [1, 2, 3, 6, 12, 24, 48, 168]:
    meta_cols[f'OZONE_lag_{lag}'] = 'float32'

# Add rolling features to meta
for window in [3, 6, 12, 24, 168]:
    meta_cols[f'OZONE_rolling_mean_{window}'] = 'float32'
    meta_cols[f'OZONE_rolling_std_{window}'] = 'float32'
    meta_cols[f'OZONE_rolling_min_{window}'] = 'float32'
    meta_cols[f'OZONE_rolling_max_{window}'] = 'float32'
    meta_cols[f'OZONE_rolling_range_{window}'] = 'float32'

# Add diff features to meta
for period in [1, 24, 168]:
    meta_cols[f'OZONE_diff_{period}'] = 'float32'

# Add other features to meta
meta_cols['OZONE_velocity'] = 'float32'
meta_cols['OZONE_acceleration'] = 'float32'
meta_cols['OZONE_expanding_mean'] = 'float32'
meta_cols['OZONE_expanding_std'] = 'float32'
meta_cols['OZONE_site_p25'] = 'float32'
meta_cols['OZONE_site_p50'] = 'float32'
meta_cols['OZONE_site_p75'] = 'float32'
meta_cols['OZONE_site_p90'] = 'float32'
meta_cols['OZONE_above_site_median'] = 'int8'
meta_cols['OZONE_above_site_p75'] = 'int8'

meta_df = pd.DataFrame({col: pd.Series(dtype=dtype) for col, dtype in meta_cols.items()})

# Apply the feature engineering function to each site
with ProgressBar():
    ddf_features = ddf.groupby('SITE_ID').apply(
        process_site_features,
        meta=meta_df
    ).reset_index(drop=True)

# Step 5: Merge with site metadata
print("\nStep 5: Merging with site metadata...")
if not df_site.empty:
    site_features = ['SITE_ID', 'LATITUDE', 'LONGITUDE', 'ELEVATION', 
                     'LAND_USE', 'TERRAIN', 'STATE']
    available_site_features = [f for f in site_features if f in df_site.columns]
    
    if available_site_features:
        df_site_subset = df_site[available_site_features]
        ddf_features = ddf_features.merge(df_site_subset, on='SITE_ID', how='left')
        print(f"   Merged {len(available_site_features)} site features")

# Step 6: Save to parquet
print("\nStep 6: Saving to parquet...")
output_path = os.path.join(OUTPUT_DIR, 'features_engineered')

with ProgressBar():
    ddf_features.to_parquet(
        output_path,
        engine='pyarrow',
        compression='snappy',
        write_index=False
    )

print(f"   Saved to: {output_path}")

# =============================================================================
# VERIFICATION AND SUMMARY
# =============================================================================
print("\n" + "=" * 80)
print("VERIFICATION")
print("=" * 80)

# Load back a sample to verify
print("\nLoading sample to verify...")
ddf_verify = dd.read_parquet(output_path)

print(f"\nFinal dataset info:")
print(f"   Total columns: {len(ddf_verify.columns)}")
print(f"   Total rows: {len(ddf_verify):,}")

print(f"\nColumn list:")
for i, col in enumerate(ddf_verify.columns):
    print(f"   {i+1:3d}. {col}")

# Show sample
print("\nSample of data:")
print(ddf_verify.head(10))

# Save feature list
feature_list_file = os.path.join(PROCESSED_DIR, 'feature_list_dask.txt')
with open(feature_list_file, 'w') as f:
    f.write("FEATURE LIST (Dask Processing)\n")
    f.write("=" * 80 + "\n\n")
    f.write(f"Total Features: {len(ddf_verify.columns)}\n")
    f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
    
    for col in ddf_verify.columns:
        f.write(f"  • {col}\n")

print(f"\nFeature list saved to: {feature_list_file}")
print("\n" + "=" * 80)
print("FEATURE ENGINEERING COMPLETE!")
print("=" * 80)

Input Directory: C:\Users\snehi\OneDrive - California State University Chico\MATH699P\Data\processed_data
Output Directory: C:\Users\snehi\OneDrive - California State University Chico\MATH699P\Data\processed_data\features_parquet

Loading processed data...
--------------------------------------------------------------------------------
Loaded ozone data: (21667030, 10)
Loaded site metadata: (159, 19)

Preprocessing...
--------------------------------------------------------------------------------
Dropped unnecessary columns. New shape: (21667030, 4)
Memory usage after downcast: 1.54 GB

STARTING FEATURE ENGINEERING WITH DASK

Step 1: Adding temporal features...
   Added temporal features. Shape: (21667030, 23)

Step 2: Sorting data...

Step 3: Converting to Dask DataFrame...
   Created Dask DataFrame with 43 partitions

Step 4: Engineering site-level features (lag, rolling, diff, etc.)...
   This may take a while...

Step 5: Merging with site metadata...
   Merged 7 site features

Ste